### CPU·NPU 벤치마크
- 파일 : n2-3_benchmark.ipynb
- 역할 : 동일 조건 벤치마크 데이터를 불러와 CPU와 NPU, 정밀도별 성능을 비교 분석합니다.
- 기능 :
    1. 실측 데이터 불러오기  
    2. 지연시간·처리량·메모리·에너지 요약  
    3. 조건별 지연시간 분포 시각화  
    4. Colab CPU에서 정밀도별 측정  


>> [전제] [코드 6-4]의 Colab 자기 측정에는 fp32·fp16·int8 ONNX와 measure()가 필요합니다. 아래 [준비] 셀이 세션에 없는 것을 즉석에서 만들며, 모델 경량화 실습의 파일을 업로드해 두면 그것을 그대로 씁니다.

>> [준비] 패키지·측정 함수·ONNX 파일 준비 (노트북 전용 셀)

In [ ]:
#- [코드 6-4]가 쓰는 것들: 패키지, measure(), fp32·int8 ONNX 파일
!pip install -q onnx onnxruntime onnxconverter-common "protobuf<6"

# - 모듈 로딩
import os, time
import numpy as np
import torch
from torchvision import models

#- 측정 함수 (추론 파이프라인 실습과 동일)
def measure(fn, n_warmup=10, n_runs=100):
    for _ in range(n_warmup):
        fn()
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    t = np.array(times)
    return t.mean(), np.median(t), np.percentile(t, 95), t.max()

#- fp32 ONNX: 없으면 즉석 변환
if not os.path.exists("mobilenet_v2.onnx"):
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1).eval()
    torch.onnx.export(m, torch.randn(1, 3, 224, 224), "mobilenet_v2.onnx", opset_version=13,
                      input_names=["input"], output_names=["output"], dynamo=False)

#- fp16 ONNX: 없으면 즉석 변환
if not os.path.exists("mobilenet_v2_fp16.onnx"):
    import onnx
    from onnxconverter_common import float16
    m16 = float16.convert_float_to_float16(onnx.load("mobilenet_v2.onnx"))
    onnx.save(m16, "mobilenet_v2_fp16.onnx")

#- int8 ONNX: 없으면 무작위 Calibration으로 즉석 양자화 (속도 측정용이라 정확도와 무관)
if not os.path.exists("mobilenet_v2_int8.onnx"):
    from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType
    class RandReader(CalibrationDataReader):
        def __init__(self, n=16):
            self.data = iter([{"input": np.random.rand(1, 3, 224, 224).astype(np.float32)}
                              for _ in range(n)])
        def get_next(self):
            return next(self.data, None)
    quantize_static("mobilenet_v2.onnx", "mobilenet_v2_int8.onnx", RandReader(),
                    quant_format=QuantFormat.QDQ, per_channel=True,
                    activation_type=QuantType.QInt8, weight_type=QuantType.QInt8)
print("준비 완료:", os.path.exists("mobilenet_v2.onnx"),
      os.path.exists("mobilenet_v2_fp16.onnx"), os.path.exists("mobilenet_v2_int8.onnx"))

# 확인 포인트 ------------------------------------------
# - 준비 완료: True True True 출력 체크
# - 경량화 실습의 int8 파일을 세션에 업로드해 두면 그 파일을 그대로 사용 (즉석 생성 생략)

>> [코드 6-1] 실측 데이터 불러오기 (n2-3_benchmark.ipynb, STEP 1)
<hr>

In [ ]:
#- 모듈 로딩
import pandas as pd

#- 실습 저장소 받기
!git clone -q https://github.com/npu-ondevice-ai/npu-course.git

#- 파일 상수: 기준 장비 실측 CSV
DATA_FILE = "npu-course/data/bench/bench_mobilenet_v2.csv"
#- 기준 장비 실측 데이터 CSV 로딩
df = pd.read_csv( DATA_FILE )
print(df.head())
print(df.groupby(["device", "precision"]).size())

# 확인 포인트 -------------------------------------------------------------------------
# - 조건마다 100회 측정 여부 체크

>> [코드 6-2] 지연시간·처리량·메모리·에너지 요약 (STEP 2)
<hr>

In [ ]:
#- 조건(장비×정밀도)별 요약 지표: p95·p99·최대·FPS·메모리·에너지
def summarize(g):
    s = g["latency_ms"]
    return pd.Series({
        "평균(ms)": s.mean(), "p95(ms)": s.quantile(0.95),
        "p99(ms)": s.quantile(0.99), "최대(ms)": s.max(),
        "FPS": 1000 / s.mean(),
        "메모리(MB)": g["mem_mb"].max(),
        "에너지(mJ)": (g["power_w"] * s).mean(),     # W × ms = mJ
    })

#- 요약표 생성
# 그룹 열을 제외한 측정 열만 골라 요약 (판 호환)
summary = df.groupby(["device", "precision"]).apply(summarize).round(2)
summary

# 확인 포인트 -------------------------------------------------------------------------
# - 조건별 요약표 출력 체크

>> [코드 6-3] 조건별 지연시간 분포 시각화 (STEP 3)
<hr>

In [ ]:
#- 모듈로딩
import matplotlib.pyplot as plt

#- 조건별 분포를 상자그림으로 (세로축 로그)
fig, ax = plt.subplots(figsize=(8, 4))
groups = [(k, g["latency_ms"].values) for k, g in df.groupby(["device", "precision"])]
ax.boxplot([v for _, v in groups], tick_labels=[f"{d}-{p}" for (d, p), _ in groups])
ax.set_ylabel("Latency (ms)"); ax.set_yscale("log")
plt.show()
# 확인 포인트 --------------------------------------------------------------------------
# - 상자 위치(중앙값)와 위로 튀는 점(꼬리 지연)의 유무 확인

>> [코드 6-4] Colab CPU에서 정밀도별 측정 (STEP 4)
<hr>

In [ ]:
#- 모듈로딩과 측정 함수 (추론 파이프라인과 동일)
import time, numpy as np, onnxruntime as ort
def measure(fn, n_warmup=10, n_runs=100):
    for _ in range(n_warmup): fn()
    t = []
    for _ in range(n_runs):
        t0 = time.perf_counter(); fn(); t.append((time.perf_counter() - t0) * 1000)
    return np.array(t)

#- 세 정밀도를 같은 조건으로 측정
rows = []
for tag, dt in [("fp32", np.float32), ("fp16", np.float16), ("int8", np.float32)]:
    # 정밀도별 파일 선택
    path = "mobilenet_v2.onnx" if tag == "fp32" else f"mobilenet_v2_{tag}.onnx"
    s = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    x = np.random.rand(1, 3, 224, 224).astype(dt)

    # 100회 측정값을 행으로 축적
    for v in measure(lambda: s.run(None, {"input": x})):
        rows.append({"device": "colab-cpu", "precision": tag, "latency_ms": v})

#- 요약 통계
mine = pd.DataFrame(rows)
print(mine.groupby("precision")["latency_ms"].describe(percentiles=[0.95, 0.99]).round(2))

# 확인 포인트 -------------------------------------------------------------------------
# - 절댓값은 배정받은 Colab 머신마다 다름: 정밀도 간 비율로 비교